# 10 — Final Benchmark on Test Set

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, joblib, torch, matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from src.data import load_processed
from src.models.ft_transformer import FTTransformer
from src.models.mlp import MLPRegressor
from src.models.pinn import PINNRegressor
from src.train import predict_torch
from src.evaluate import regression_metrics
from src.config import MODELS_DIR, FIGURES_DIR

In [ ]:
X_train, y_train = load_processed('train')
X_test, y_test = load_processed('test')
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
input_dim = X_train.shape[1]

In [ ]:
mlp = MLPRegressor(input_dim); mlp.load_state_dict(torch.load(MODELS_DIR / 'mlp.pt', map_location='cpu'))
ftt = FTTransformer(input_dim); ftt.load_state_dict(torch.load(MODELS_DIR / 'ft_transformer.pt', map_location='cpu'))
pinn = PINNRegressor(input_dim); pinn.load_state_dict(torch.load(MODELS_DIR / 'pinn.pt', map_location='cpu'))
xgb = joblib.load(MODELS_DIR / ('XGBoost_tuned.joblib' if (MODELS_DIR / 'XGBoost_tuned.joblib').exists() else 'XGBoost.joblib'))
lgbm = joblib.load(MODELS_DIR / 'LightGBM.joblib')
cb = joblib.load(MODELS_DIR / 'CatBoost.joblib')
stk = joblib.load(MODELS_DIR / 'stacker.joblib')

In [ ]:
preds = {
    'Ridge': ridge.predict(X_test),
    'XGBoost': xgb.predict(X_test),
    'LightGBM': lgbm.predict(X_test),
    'CatBoost': cb.predict(X_test),
    'MLP': predict_torch(mlp, X_test),
    'FT-Transformer': predict_torch(ftt, X_test),
    'PINN': predict_torch(pinn, X_test),
}
base = {'xgb': preds['XGBoost'], 'lgbm': preds['LightGBM'], 'cb': preds['CatBoost'], 'ftt': preds['FT-Transformer']}
preds['Stacking'] = stk.predict(base)
rows = [dict(model=k, **regression_metrics(y_test, v)) for k, v in preds.items()]
bench = pd.DataFrame(rows).set_index('model').sort_values('RMSE')
bench

In [ ]:
bench.to_csv(FIGURES_DIR / 'benchmark.csv')
fig, ax = plt.subplots(figsize=(8,5))
bench['RMSE'].plot.barh(ax=ax, color='steelblue'); ax.set_title('Test RMSE per model (lower is better)')
plt.tight_layout(); plt.savefig(FIGURES_DIR / '10_benchmark_rmse.png', dpi=120); plt.show()

In [ ]:
best_name = bench.index[0]; best_pred = preds[best_name]
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(y_test[:5000], best_pred[:5000], s=4, alpha=0.4); ax[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax[0].set_xlabel('True'); ax[0].set_ylabel('Pred'); ax[0].set_title(f'{best_name}: True vs Pred')
resid = y_test - best_pred
ax[1].hist(resid, bins=80, color='teal'); ax[1].set_title(f'{best_name}: Residuals')
plt.tight_layout(); plt.savefig(FIGURES_DIR / '10_residuals.png', dpi=120); plt.show()